# 16 · Electrochemical thermodynamics

A fuel cell turns the Gibbs energy of a reaction directly into electrical work; an electrolyser runs the
same reaction backwards; a battery stores it. The link is Faraday's law: $\Delta G = -nFE$. This notebook
computes the reversible voltages, efficiencies and energy densities that bound every electrochemical energy
technology - fuel cells, hydrogen electrolysis and batteries.

**What you will learn**
- relate Gibbs energies to cell voltages
- apply the Nernst equation and the temperature coefficient
- compute fuel-cell and electrolyser efficiencies and hydrogen electricity consumption
- estimate theoretical battery energy densities

**Before you start:** notebooks 14-15  ·  **Time:** about 60 min

In [1]:
try:                      # on Google Colab (or anywhere engthermo is missing): install it from GitHub
    import engthermo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engthermo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engthermo import electrochem, reaction
from engthermo.constants import F, R
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## The hydrogen fuel cell
H$_2$ + ½O$_2$ -> H$_2$O(l) transfers two electrons per molecule of hydrogen:

In [2]:
h2 = {"H2": -1, "O2": -0.5, "H2O(l)": 1}
lim = electrochem.fuel_cell_limits(h2, 2)
print(f"Delta G° = {lim['dG']/1e3:.1f} kJ/mol, Delta H° = {lim['dH']/1e3:.1f} kJ/mol")
print(f"reversible cell voltage E° = {lim['E_rev']:.3f} V; thermoneutral voltage Delta H/(nF) = {lim['E_thermoneutral']:.3f} V")
print(f"maximum efficiency Delta G / Delta H = {lim['efficiency_max']:.1%}  (a heat engine at the same 25 degC would deliver nothing)")

Delta G° = -237.1 kJ/mol, Delta H° = -285.8 kJ/mol
reversible cell voltage E° = 1.229 V; thermoneutral voltage Delta H/(nF) = 1.481 V
maximum efficiency Delta G / Delta H = 83.0%  (a heat engine at the same 25 degC would deliver nothing)


The reversible voltage of 1.23 V is the fuel cell's ceiling; the 83 % "thermodynamic efficiency" is the
fraction of the heating value available as work - a limit not set by Carnot, because no heat engine is
involved. Real cells run at 0.6-0.8 V under load, and the difference appears as heat:

In [3]:
for V in (1.0, 0.8, 0.7, 0.6):
    print(f"cell voltage {V:.1f} V: efficiency (HHV basis) {electrochem.efficiency_at_voltage(V, lim['E_thermoneutral']):.1%}, "
          f"heat released {(lim['E_thermoneutral'] - V) * 2 * F / 1e3:.0f} kJ per mol H2")

cell voltage 1.0 V: efficiency (HHV basis) 67.5%, heat released 93 kJ per mol H2
cell voltage 0.8 V: efficiency (HHV basis) 54.0%, heat released 131 kJ per mol H2
cell voltage 0.7 V: efficiency (HHV basis) 47.3%, heat released 151 kJ per mol H2
cell voltage 0.6 V: efficiency (HHV basis) 40.5%, heat released 170 kJ per mol H2


## Temperature and pressure

In [4]:
for T in (298.15, 353.15, 1073.15):
    rxn = h2 if T < 373 else {"H2": -1, "O2": -0.5, "H2O": 1}          # above 100 degC the product is steam
    print(f"{T-273.15:5.0f} degC: E° = {electrochem.standard_potential(rxn, 2, T):.3f} V, max efficiency {electrochem.fuel_cell_limits(rxn, 2, T)['efficiency_max']:.1%}")
print(f"temperature coefficient at 25 degC: {electrochem.temperature_coefficient(h2, 2)*1e3:.2f} mV/K")
E0 = lim["E_rev"]
for p_H2, p_O2 in ((1, 1), (3, 3), (1, 0.21)):
    Q = 1 / (p_H2 * p_O2**0.5)
    print(f"p_H2 = {p_H2} bar, p_O2 = {p_O2} bar: E = {electrochem.nernst(E0, 2, Q):.3f} V")

   25 degC: E° = 1.229 V, max efficiency 83.0%
   80 degC: E° = 1.183 V, max efficiency 80.4%
  800 degC: E° = 0.977 V, max efficiency 75.9%
temperature coefficient at 25 degC: -0.85 mV/K
p_H2 = 1 bar, p_O2 = 1 bar: E = 1.229 V
p_H2 = 3 bar, p_O2 = 3 bar: E = 1.250 V
p_H2 = 1 bar, p_O2 = 0.21 bar: E = 1.219 V


The reversible voltage falls with temperature (the reaction's entropy is negative), so high-temperature
fuel cells give up a little voltage but gain fast kinetics and tolerance to carbon monoxide. Pressurising
the reactants raises it slightly (the Nernst equation); running on air instead of oxygen costs about 10 mV.

## Water electrolysis: making green hydrogen

In [5]:
print(f"minimum (reversible) voltage {lim['E_rev']:.3f} V; thermoneutral voltage {lim['E_thermoneutral']:.3f} V")
for V in (1.48, 1.7, 1.9, 2.1):
    eta = electrochem.electrolyser_efficiency(V, lim["E_thermoneutral"])
    kwh = V * 2 * F / 2.016e-3 / 3.6e6
    print(f"cell voltage {V:.2f} V: efficiency (HHV) {eta:.1%}, electricity {kwh:.1f} kWh per kg H2")

minimum (reversible) voltage 1.229 V; thermoneutral voltage 1.481 V
cell voltage 1.48 V: efficiency (HHV) 100.1%, electricity 39.4 kWh per kg H2
cell voltage 1.70 V: efficiency (HHV) 87.1%, electricity 45.2 kWh per kg H2
cell voltage 1.90 V: efficiency (HHV) 78.0%, electricity 50.5 kWh per kg H2
cell voltage 2.10 V: efficiency (HHV) 70.5%, electricity 55.8 kWh per kg H2


Between the reversible and the thermoneutral voltage an electrolyser absorbs heat from its surroundings
(the entropy term); above it, it releases heat. Practical cells run at 1.7-2.1 V, needing 45-55 kWh per kg
of hydrogen against a heating value of 39.4 kWh/kg - a 70-85 % efficiency. Combined with the fuel cell's
50-60 %, a hydrogen "round trip" returns about 40 % of the electricity: thermodynamics, not engineering
immaturity, is the largest part of that loss.

## Batteries: theoretical energy densities
The maximum energy a cell can store per kilogram of active materials follows from $\Delta G$ and the masses:

In [6]:
reaction.EXTRA_FORMATION["ZnO"] = (-350.5e3, -320.5e3, 40.3)      # JANAF (solid), Cp 40.3 J/(mol K)
cells = {"zinc-air: 2 Zn + O2 -> 2 ZnO": ({"O2": -1, "ZnO": 2}, 4, 2 * 65.38e-3),
         "hydrogen-oxygen (fuel cell): H2 + 1/2 O2 -> H2O": (h2, 2, 2.016e-3 + 16.0e-3)}
for name, (rxn, n_e, mass) in cells.items():
    dG = reaction.standard_gibbs(rxn) if "Zn" not in name else 2 * (-320.5e3)
    print(f"{name:<52}: E° = {-dG/(n_e*F):.2f} V, theoretical specific energy {electrochem.energy_per_mass(dG, mass):.0f} Wh/kg")
print("for comparison (from tabulated cell reactions): lead-acid E° 2.04 V, about 170 Wh/kg; lithium-ion about 400-500 Wh/kg theoretical")

zinc-air: 2 Zn + O2 -> 2 ZnO                        : E° = 1.66 V, theoretical specific energy 1362 Wh/kg
hydrogen-oxygen (fuel cell): H2 + 1/2 O2 -> H2O     : E° = 1.23 V, theoretical specific energy 3656 Wh/kg
for comparison (from tabulated cell reactions): lead-acid E° 2.04 V, about 170 Wh/kg; lithium-ion about 400-500 Wh/kg theoretical


Theoretical specific energies are upper bounds: practical cells reach a quarter to a third of them once
electrolyte, current collectors and casing are added. The ranking survives - metal-air and hydrogen systems
have the highest thermodynamic ceilings because oxygen from the air is a "free" reactant - and it explains
why lithium (the lightest metal with a high potential) dominates portable storage.

## Inside the algorithm: Faraday's law
Two electrons per hydrogen molecule; the electrical work per mole is $nFE$, and it cannot exceed $-\Delta G$:

In [7]:
dG = reaction.standard_gibbs(h2)
print(f"E° = -Delta G / (n F) = {-dG / (2 * F):.4f} V;  library: {electrochem.standard_potential(h2, 2):.4f} V")
print(f"electrical work per mol H2 at E°: {2 * F * (-dG / (2 * F)) / 1e3:.1f} kJ = -Delta G")

E° = -Delta G / (n F) = 1.2288 V;  library: 1.2288 V
electrical work per mol H2 at E°: 237.1 kJ = -Delta G


## Exercises
1. A solid-oxide fuel cell at 800 degC runs on hydrogen and produces steam. Compute its reversible voltage and maximum efficiency, and compare with a 25 degC cell. Which loses more of the heating value as reversible heat?
2. Round trip: an electrolyser at 1.85 V feeds hydrogen to a fuel cell at 0.72 V. What fraction of the electricity returns? How does this compare with a pumped-hydro or battery storage round trip of 75-90 %?
3. **Implement it yourself:** write the Nernst equation for the hydrogen cell as a function of p_H2, p_O2 and T, and plot E against temperature from 25 to 200 degC at 1 bar (use the steam reaction above 100 degC).